# Test Parse - Mapeo de Carreras y Parseo Vectorizado de Nota Final

Este notebook realiza las siguientes operaciones:
1. Carga del dataset `reglamento_nuevo_unificado.csv`.
2. Mapeo de carreras universitarias según el código de materia/activo.
3. Extracción vectorizada mediante expresiones regulares de los intentos de finales (`1F` y `2F`).
4. Verificación y testeo de la correcta lectura de datos.

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Carga del dataset
csv_path = 'reglamento_nuevo_unificado.csv'
if not os.path.exists(csv_path):
    csv_path = os.path.join('proyectos_ML', 'Primer_Parcial', 'reglamento_nuevo_unificado.csv')

df_raw = pd.read_csv(csv_path)
print(f"Dataset cargado: {df_raw.shape[0]:,} filas y {df_raw.shape[1]} columnas.")

# 2. Mapeo de Carreras
career_code_mapping = {
    'CIV-PLS13': 'Ing. Civil', 'CIV-PLS23': 'Ing. Civil', 'INT9CONSTR': 'Ing. Civil',
    'INT9TRANSP': 'Ing. Civil', 'INT9ORTERR': 'Ing. Civil', 'INT9SANEHI': 'Ing. Civil',
    'ELE-PLS13': 'Ing. Electromecánica', 'ELE-PLS23': 'Ing. Electromecánica',
    'INT9ELECTR': 'Ing. Electromecánica', 'INT9SDIGYT': 'Ing. Electromecánica',
    'MCT-PLS13': 'Ing. Mecatrónica', 'MCT-PLS23': 'Ing. Mecatrónica', 'MCT9-OPT': 'Ing. Mecatrónica',
    'IND-PLS13': 'Ing. Industrial', 'IND-PLS23': 'Ing. Industrial',
    'INT9G-ECO': 'Ing. Industrial', 'INT9-PROYT': 'Ing. Industrial',
    'CGF-PLS13': 'Ing. Geográfica', 'CGF-PLS23': 'Ing. Geográfica', 'INT9RNYMA': 'Ing. Geográfica',
    'MEC-PLS13': 'Ing. Mecánica', 'MEC-PLS23': 'Ing. Mecánica',
    'INT9MECANI': 'Ing. Mecánica', 'MEC9-OPT': 'Ing. Mecánica',
    'ECA-PLS13': 'Ing. Electrónica', 'ECA-PLS23': 'Ing. Electrónica', 'ECA9-OPT': 'Ing. Electrónica'
}

df_clean = df_raw.copy()
df_clean['Carrera'] = df_clean['Firma'].astype(str).str.strip().map(career_code_mapping)

# 3. Parsing Vectorizado de Nota.Final
s_nota = df_clean['Nota.Final'].fillna('')

# 1º Final (1F)
df_clean['Rendio_1F'] = s_nota.str.contains('1F-').astype(int)
df_clean['Nota_1F'] = s_nota.str.extract(r'1F-(\d)')[0].astype(float)
df_clean['Aprobo_1F'] = (df_clean['Nota_1F'] >= 2).astype(int)

# 2º Final (2F)
df_clean['Rendio_2F'] = s_nota.str.contains('2F-').astype(int)
df_clean['Nota_2F'] = s_nota.str.extract(r'2F-(\d)')[0].astype(float)
df_clean['Aprobo_2F'] = (df_clean['Nota_2F'] >= 2).astype(int)

# Nota Numérica Final / Última nota registrada
df_clean['Nota_Num'] = (
    df_clean['Nota.Final']
    .astype(str)
    .str.extractall(r'(\d+)')
    [0]
    .groupby(level=0)
    .last()
    .astype(float)
)
df_clean.loc[df_clean['Nota.Final'].isna(), 'Nota_Num'] = np.nan
df_clean['Aprobo_Cualquiera'] = (df_clean['Nota_Num'] >= 2).astype(int)

# 4. Verificación y Resultados (Test Parse)
print("\n=== RESULTADOS DE VERIFICACIÓN / TEST PARSE ===")
print(f"Total de registros analizados: {len(df_clean):,}")
print(f"- Rendieron 1F: {df_clean['Rendio_1F'].sum():,} | Aprobados en 1F: {df_clean['Aprobo_1F'].sum():,}")
print(f"- Rendieron 2F: {df_clean['Rendio_2F'].sum():,} | Aprobados en 2F: {df_clean['Aprobo_2F'].sum():,}")
print(f"- Fueron directo a 2F (sin 1F): {((df_clean['Rendio_1F'] == 0) & (df_clean['Rendio_2F'] == 1)).sum():,}")
print(f"- Aprobaron la materia (Cualquiera): {df_clean['Aprobo_Cualquiera'].sum():,}")

# Muestra de validación
columnas_test = ['Nota.Final', 'Rendio_1F', 'Nota_1F', 'Aprobo_1F', 'Rendio_2F', 'Nota_2F', 'Aprobo_2F', 'Nota_Num']
print("\n--- Muestra aleatoria de verificación de parseo ---")
print(df_clean[columnas_test].dropna(subset=['Nota.Final']).sample(10, random_state=42))